## Import libraries

In [ ]:
import numpy as np
from itertools import combinations, product

## Input Evaluation Function

In [ ]:
def evaluation(matrix):
    if not isinstance(matrix, (list, np.ndarray)):
        raise ValueError("Input must be a list or a NumPy ndarray.")

    try:
        matrix = np.asarray(matrix)
    except (TypeError, ValueError):
        raise ValueError("Input must be a rectangular 2D matrix.")

    if matrix.size == 0:
        raise ValueError("Matrix is empty.")

    if matrix.ndim != 2:
        raise ValueError("Matrix must be 2-dimensional.")

    if not (np.issubdtype(matrix.dtype, np.integer) or
            np.issubdtype(matrix.dtype, np.floating)):
        raise ValueError("Matrix entries must be real numeric values.")

    if not np.all(np.isin(matrix, [-1, 0, 1])):
        raise ValueError("A TU matrix can only contain -1, 0, and 1.")

    return matrix

## Generate Submatrices

In [ ]:
def submatrix(matrix):
    rows_count, columns_count = matrix.shape
    max_size = min(rows_count, columns_count)

    for size in range(1, max_size + 1):
        row_combinations = combinations(range(rows_count), size)
        column_combinations = combinations(range(columns_count), size)

        for rows, columns in product(row_combinations, column_combinations):
            yield matrix[np.ix_(rows, columns)]

## Calculate the Determinant

In [48]:
def determinant_laplace(matrix):

    if hasattr(matrix, "tolist"):
        matrix = matrix.tolist()

    n = len(matrix)

    if n == 1:
        return matrix[0][0]

    if n == 2:
        return (matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0])

    determinant = 0
    for j in range(n):
        minor = [[matrix[i][k] for k in range(n) if k != j] for i in range(1, n)]

        cofactor = (-1) ** j * matrix[0][j]

        determinant += cofactor * determinant_laplace(minor)

    return determinant


## Executor Function

In [49]:
def is_totally_unimodular(matrix, method="laplace"):
    evaluated = evaluation(matrix)

    if method == "bareiss":
        determinant_function = determinant_bareiss
    elif method == "laplace":
        determinant_function = determinant_laplace
    else:
        raise ValueError("method must be 'bareiss' or 'laplace'.")

    determinant_runs = 0

    for sub in submatrix(evaluated):
        determinant = determinant_function(sub)
        determinant_runs += 1

        if determinant not in (-1, 0, 1):
            return {
                "result": "This matrix is not Totally Unimodular",
                "determinant_runs": determinant_runs
            }

    return {
        "result": "This matrix is Totally Unimodular",
        "determinant_runs": determinant_runs
    }


## Example

In [50]:
A1 = [
    [-1,0,0],
    [1,1,0],
    [0,-1,-1]
]

A2 = [
    [1,1,1,0,0,0,0],
    [0,1,1,1,0,0,0],
    [0,0,1,1,1,0,0],
    [0,0,0,1,1,1,0],
    [0,0,0,0,1,1,1]
]

A3 = [
    [1,1,0,1],    
    [1,2,0,1],    
    [0,0,0,1],    
    [0,0,1,0]    
]

A4 = [
    [1,0,1,-1],
    [1,1,0,-1],
    [0,-1,-1,0],
    [0,1,1,0],
]

In [51]:
result = is_totally_unimodular(A1)
print("A1:", result["result"])
print("Determinants calculated:", result["determinant_runs"])

A1: This matrix is Totally Unimodular
Determinants calculated: 19


In [52]:
result = is_totally_unimodular(A2)
print("A2:", result["result"])
print("Determinants calculated:", result["determinant_runs"])

A2: This matrix is Totally Unimodular
Determinants calculated: 791


In [53]:
try:
    result = is_totally_unimodular(A3)
    print("A3:", result["result"])
    print("Determinants calculated:", result["determinant_runs"])
except ValueError as e:
    print("A3:", e)

A3: A TU matrix can only contain -1, 0, and 1.


In [54]:
result = is_totally_unimodular(A4)
print("A4:", result["result"])
print("Determinants calculated:", result["determinant_runs"])

A4: This matrix is not Totally Unimodular
Determinants calculated: 53


## Testing Area

In [ ]:
M = [[0]]

result = is_totally_unimodular(M)
print("Test_Matrix:", result["result"])
print("Determinants calculated:", result["determinant_runs"])

## Ask ChatGPT to test the algorithm

In [ ]:
# A 1x1 TU matrix
result = is_totally_unimodular([[-1]])
assert result["result"] == "This matrix is Totally Unimodular"
assert result["determinant_runs"] == 1

# Identity matrix is TU
result = is_totally_unimodular([[1, 0], [0, 1]])
assert result["result"] == "This matrix is Totally Unimodular"
assert result["determinant_runs"] == 5

# This matrix has determinant -2, so it is not TU.
test_matrix = [
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
]
result = is_totally_unimodular(test_matrix)
assert result["result"] == "This matrix is not Totally Unimodular"
assert result["determinant_runs"] == 19

# Both determinant methods should provide the same TU decision.
for method in ("bareiss", "laplace"):
    result = is_totally_unimodular(test_matrix, method=method)
    assert result["result"] == "This matrix is not Totally Unimodular"

# Invalid method must be rejected.
try:
    is_totally_unimodular([[1]], method="unknown")
except ValueError:
    pass
else:
    raise AssertionError("Invalid determinant method was not rejected.")

# Invalid values must be rejected.
try:
    evaluation([[1, 2], [0, 1]])
except ValueError:
    pass
else:
    raise AssertionError("Invalid matrix was not rejected.")

# Empty and non-2D inputs must be rejected.
for invalid_matrix in ([], [1, 0, -1]):
    try:
        evaluation(invalid_matrix)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input was not rejected.")

print("All tests passed.")

## bareiss method for determinant (Not ready)

In [ ]:
def determinant_bareiss(matrix):
    # Bareiss algorithm: exact determinant for integer matrices
    matrix = [[int(value) for value in row] for row in matrix.tolist()]

    n = len(matrix)

    if n == 1:
        return matrix[0][0]

    sign = 1
    previous_pivot = 1

    for k in range(n - 1):

        # If the pivot is zero, find a lower row to swap with it.
        if matrix[k][k] == 0:
            swap_row = None

            for row in range(k + 1, n):
                if matrix[row][k] != 0:
                    swap_row = row
                    break

            if swap_row is None:
                return 0

            matrix[k], matrix[swap_row] = matrix[swap_row], matrix[k]
            sign *= -1

        pivot = matrix[k][k]

        for i in range(k + 1, n):
            for j in range(k + 1, n):
                numerator = (
                    matrix[i][j] * pivot
                    - matrix[i][k] * matrix[k][j]
                )
                matrix[i][j] = numerator // previous_pivot

        for i in range(k + 1, n):
            matrix[i][k] = 0

        previous_pivot = pivot

    return sign * matrix[n - 1][n - 1]